In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel,EmailStr 
 app = FastAPI()

In [ ]:
class UserIn(BaseModel):
    username: str
    password: str
    email: EmailStr
    full_name : str | None = None 


class UserOut(BaseModel):
    username: str
    email: EmailStr
    full_name: str | None = None

class UserInDB(BaseModel):
    username: str
    hashed_password: str 
    email: EmailStr
    full_name: str | None = None

def fake_password_hasher(raw_password: str):
    return "supersecret"+ raw_password


def fake_save_user(user_in: UserIn):
    hashed_password = fake_password_hasher(user_in.password)
    # **user_in.model_dump() 解包后可以添加额外的参数
    user_in_db = UserInDB(**user_in.model_dump(),hashed_password = hashed_password)
    print("User saved .. not really")
    return user_in_db

@app.get("/user/",response_model=UserOut)
def create_user(user_in: UserIn):
    user_saved = fake_save_user(user_in)
    return user_saved

In [ ]:
# 抽出基本类，简化代码

class UserBase(BaseModel):
    username: str
    email: EmailStr
    full_name: str | None = None

class UserIn(UserBase):
    password: str 

class UserOut(UserBase):
    pass


class UserInDb(UserBase):
    hashed_password: str

def fake_password_hasher(raw_password: str):
    return "supersecret"+ raw_password


def fake_save_user(user_in: UserIn):
    hashed_password = fake_password_hasher(user_in.password)
    # **user_in.model_dump() 解包后可以添加额外的参数
    user_in_db = UserInDB(**user_in.model_dump(),hashed_password = hashed_password)
    print("User saved .. not really")
    return user_in_db

@app.get("/user/",response_model=UserOut)
def create_user(user_in: UserIn):
    user_saved = fake_save_user(user_in)
    return user_saved

In [ ]:
# Union 响应是其中之一

class BaseItem(BaseModel):
    description: str
    type: str 

class CartItem(BaseItem):
    type: str = 'car'

class PlaneItem(BaseItem):
    type: str = 'plane'
    size : int 


items ={
    'item1':{'description':'All my friends drive a low rider ','type': 'cat'},
    'item2':{
        'description':'Music is my aerplane , it is ly aeroplane',
        'type':'plane',
        'size': 5
    }
}

@app.get("/items/{item_id}",response_model=Union[PlaneItem | CartItem])
def read_item(item_id: str):
    return items[item_id]